# KV cache & sampling from scratch
**LLMs From Basics · M1.5 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/m1-5-kv-cache-sampling.ipynb)

You'll take a tiny (random, untrained) decoder-only transformer written in NumPy and:

1. generate text the **naive** way, re-running the model on everything at every step
2. add a **KV cache** and prove it produces **exactly the same** tokens, with far less work
3. compute real **KV-cache sizes** for Llama models
4. implement **temperature**, **top-k** and **top-p** sampling, and watch what they do

**How to use it:** run cells top to bottom (Shift+Enter). Each `# TODO` cell is followed by a ✅ check cell. Stuck for more than five minutes? The **Solutions** are at the bottom.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

def softmax(x, axis=-1):
    z = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def rmsnorm(x, g, eps=1e-6):
    return x / np.sqrt((x ** 2).mean(-1, keepdims=True) + eps) * g

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))

# A tiny decoder-only transformer (M1.4): learned positions, pre-RMSNorm, 2 layers, 4 heads, a separate unembedding W_U.
VOCAB = ["<bos>", "I", "sat", "by", "the", "river", "bank", "and", "watched", "water", "flow", ".",
         "a", "boat", "went", "past", "slowly", "birds", "sang", "quietly"]
V, D, H, L, MAXLEN = len(VOCAB), 32, 4, 2, 256
DK = D // H
r = np.random.default_rng(42)
P = {"tok": r.normal(size=(V, D)), "pos": 0.5 * r.normal(size=(MAXLEN, D)), "g_f": np.ones(D),
     "W_U": 0.3 * r.normal(size=(D, V)), "layers": []}
for _ in range(L):
    P["layers"].append({
        "g1": np.ones(D), "g2": np.ones(D),
        "Wq": r.normal(size=(D, D)) / np.sqrt(D), "Wk": r.normal(size=(D, D)) / np.sqrt(D),
        "Wv": r.normal(size=(D, D)) / np.sqrt(D), "Wo": r.normal(size=(D, D)) / np.sqrt(D),
        "W1": r.normal(size=(D, 4 * D)) / np.sqrt(D), "W2": r.normal(size=(4 * D, D)) / np.sqrt(4 * D)})

COUNT = {"token_passes": 0}   # how many token positions went through the model

def forward(tokens):
    # Full forward pass over a list of token ids. Returns logits of shape (n, V).
    n = len(tokens)
    COUNT["token_passes"] += n
    x = P["tok"][tokens] + P["pos"][:n]
    mask = np.where(np.triu(np.ones((n, n), dtype=bool), k=1), -np.inf, 0.0)
    for lp in P["layers"]:
        a = rmsnorm(x, lp["g1"])
        q, k, v = ((a @ lp[w]).reshape(n, H, DK).transpose(1, 0, 2) for w in ("Wq", "Wk", "Wv"))  # (H, n, DK)
        w = softmax(q @ k.transpose(0, 2, 1) / np.sqrt(DK) + mask)
        x = x + (w @ v).transpose(1, 0, 2).reshape(n, D) @ lp["Wo"]
        x = x + gelu(rmsnorm(x, lp["g2"]) @ lp["W1"]) @ lp["W2"]
    return rmsnorm(x, P["g_f"]) @ P["W_U"]

def ids(words): return [VOCAB.index(w) for w in words.split()]
def words(token_ids): return " ".join(VOCAB[t] for t in token_ids)

prompt = ids("<bos> I sat by the")
print("logits for the prompt:", forward(prompt).shape)

## 1. Naive greedy generation
The loop from the lesson: run the model on **everything so far**, take the logits at the **last** position, pick the most likely token (greedy), append, repeat.

In [ ]:
def generate_naive(prompt, n_new):
    # Returns the list prompt + n_new greedily generated token ids.
    seq = list(prompt)
    # TODO: repeat n_new times: logits = forward(seq)[-1]; append int(np.argmax(logits)) to seq
    raise NotImplementedError("Fill in generate_naive")

In [ ]:
COUNT["token_passes"] = 0
out_naive = generate_naive(prompt, 20)
naive_passes = COUNT["token_passes"]
assert len(out_naive) == len(prompt) + 20 and out_naive[:len(prompt)] == prompt
assert naive_passes == sum(range(len(prompt), len(prompt) + 20)), naive_passes
print(words(out_naive))
print("token-passes:", naive_passes)
print("✅ naive generation works (the words are random: the model is untrained)")

## 2. A KV cache
`forward_step` processes **one** token at position `pos`. For every layer it computes that token's $q, k, v$, **appends** $k$ and $v$ to the layer's cache, and lets $q$ attend over all cached keys. No mask is needed: the cache only ever contains the past.

The cache is a list with one dict per layer: `cache[l]["k"]` and `cache[l]["v"]` have shape `(H, t, DK)` after $t$ tokens.

In [ ]:
def new_cache():
    return [{"k": np.zeros((H, 0, DK)), "v": np.zeros((H, 0, DK))} for _ in range(L)]

def forward_step(token, pos, cache):
    # One token in, logits (V,) out. Updates `cache` in place.
    COUNT["token_passes"] += 1
    x = P["tok"][token] + P["pos"][pos]                   # (D,)
    for l, lp in enumerate(P["layers"]):
        a = rmsnorm(x, lp["g1"])
        q = (a @ lp["Wq"]).reshape(H, 1, DK)
        # TODO 1: compute k and v the same way, shape (H, 1, DK)
        # TODO 2: append them to the cache: cache[l]["k"] = np.concatenate([cache[l]["k"], k], axis=1), same for "v"
        # TODO 3: w = softmax(q @ K^T / sqrt(DK)) using the cached keys  -> shape (H, 1, t)
        # TODO 4: o = (w @ cached values).reshape(D); x = x + o @ lp["Wo"]
        raise NotImplementedError("Fill in forward_step")
        x = x + gelu(rmsnorm(x, lp["g2"]) @ lp["W1"]) @ lp["W2"]
    return rmsnorm(x, P["g_f"]) @ P["W_U"]

In [ ]:
cache = new_cache()
step_logits = np.stack([forward_step(t, i, cache) for i, t in enumerate(prompt)])
full_logits = forward(prompt)
print("max |difference| vs the full forward pass:", float(np.abs(step_logits - full_logits).max()))
assert np.allclose(step_logits, full_logits, atol=1e-9), "Cached step-by-step logits must equal the full pass."
assert cache[0]["k"].shape == (H, len(prompt), DK), cache[0]["k"].shape
print("✅ forward_step with a KV cache matches the full forward pass")

## 3. Cached generation
Prefill the cache with the prompt (here simply one `forward_step` per prompt token; real systems do it in one parallel pass), then generate: each step feeds in just the newest token.

In [ ]:
def generate_cached(prompt, n_new):
    cache = new_cache()
    for i, t in enumerate(prompt):                        # prefill
        logits = forward_step(t, i, cache)
    seq = list(prompt)
    # TODO: repeat n_new times: nxt = int(np.argmax(logits)); append it to seq;
    #       then logits = forward_step(nxt, len(seq) - 1, cache)
    raise NotImplementedError("Fill in generate_cached")

In [ ]:
COUNT["token_passes"] = 0
out_cached = generate_cached(prompt, 20)
cached_passes = COUNT["token_passes"]
assert out_cached == out_naive, "The cache must not change a single token."
print("identical output ✓   token-passes: naive", naive_passes, "vs cached", cached_passes)

n_new = 200
t0 = time.perf_counter(); a = generate_naive(prompt, n_new); t1 = time.perf_counter()
b = generate_cached(prompt, n_new); t2 = time.perf_counter()
assert a == b
print(f"{n_new} new tokens: naive {t1 - t0:.2f}s, cached {t2 - t1:.2f}s")
print("✅ same tokens, much less work")

## 4. How big is the cache?
$$\text{KV bytes} = 2 \times L \times n_{\text{kv}} \times d_{\text{head}} \times n_{\text{tokens}} \times b$$

In [ ]:
def kv_cache_bytes(layers, kv_heads, d_head, seq_len, bytes_per=2, batch=1):
    # TODO: return the formula above (times batch)
    raise NotImplementedError("Fill in kv_cache_bytes")

In [ ]:
GiB, KiB = 2 ** 30, 2 ** 10
assert kv_cache_bytes(32, 32, 128, 1) == 512 * KiB              # Llama 2 7B, per token
assert kv_cache_bytes(32, 32, 128, 4096) == 2 * GiB             # one full 4K sequence
assert kv_cache_bytes(32, 8, 128, 131072) == 16 * GiB            # Llama 3.1 8B (GQA) at 128K
assert kv_cache_bytes(80, 8, 128, 4096, batch=8) == 10 * GiB     # Llama 2 70B (GQA), 8 sequences
for name, cfg in {"Llama 2 7B": (32, 32, 128), "Llama 3.1 8B": (32, 8, 128), "Llama 2 70B": (80, 8, 128)}.items():
    print(f"{name:13s} {kv_cache_bytes(*cfg, 1) / KiB:6.0f} KiB/token   {kv_cache_bytes(*cfg, 4096) / GiB:5.2f} GiB at 4,096 tokens")
print("✅ cache sizes check out")

## 5. Sampling: temperature, top-k, top-p
Implement the three knobs from the lesson. Each filter takes a probability vector, zeroes the tokens it removes, and **renormalizes**.
- `apply_temperature(logits, T)`: softmax of `logits / T`
- `top_k_filter(probs, k)`: keep the `k` largest
- `top_p_filter(probs, p)`: keep the **smallest** set of top tokens whose probabilities sum to at least `p`

In [ ]:
def apply_temperature(logits, T):
    # TODO
    raise NotImplementedError

def top_k_filter(probs, k):
    # TODO: indices of the k largest: np.argsort(probs)[::-1][:k]; zero the rest; renormalize
    raise NotImplementedError

def top_p_filter(probs, p):
    # TODO: order = np.argsort(probs)[::-1]; cum = np.cumsum(probs[order])
    #       n_keep = number of tokens up to and including the first where cum >= p
    #       keep order[:n_keep]; zero the rest; renormalize
    raise NotImplementedError

def sample(logits, rng, T=1.0, k=None, p=None):
    # T == 0 means greedy. Otherwise temperature, then top-k, then top-p, then draw.
    if T == 0:
        return int(np.argmax(logits))
    probs = apply_temperature(np.asarray(logits, dtype=float), T)
    if k is not None:
        probs = top_k_filter(probs, k)
    if p is not None:
        probs = top_p_filter(probs, p)
    return int(rng.choice(len(probs), p=probs))

In [ ]:
assert np.allclose(apply_temperature(np.array([2., 1., 0.]), 1), [0.665, 0.245, 0.090], atol=1e-3)
assert np.allclose(apply_temperature(np.array([2., 1., 0.]), 0.5), [0.867, 0.117, 0.016], atol=1e-3)
assert np.allclose(apply_temperature(np.array([2., 1., 0.]), 2), [0.506, 0.307, 0.186], atol=1e-3)
q = np.array([0.5, 0.3, 0.15, 0.05])
assert np.allclose(top_k_filter(q, 2), [0.625, 0.375, 0, 0])
assert np.allclose(top_p_filter(q, 0.9), np.array([0.5, 0.3, 0.15, 0]) / 0.95)
assert np.allclose(top_p_filter(np.array([0.6, 0.3, 0.1]), 0.8), [2 / 3, 1 / 3, 0])      # lesson faded example
t05 = apply_temperature(np.log([0.6, 0.3, 0.1]), 0.5)
assert np.isclose(t05[0], 36 / 46) and np.isclose(top_p_filter(t05, 0.8)[0], 0.8)    # T = 0.5 then top-p 0.8

rng = np.random.default_rng(0)
logits = np.array([3.0, 2.2, 1.8, 1.4, 1.0, 0.6, 0.2, -0.2])   # the lesson's playground
draws = np.array([sample(logits, rng, T=1.0) for _ in range(20000)])
freq = np.bincount(draws, minlength=len(logits)) / len(draws)
assert np.allclose(freq, apply_temperature(logits, 1.0), atol=0.015), freq
assert sample(logits, rng, T=0) == 0
assert set(sample(logits, rng, T=1.5, k=3) for _ in range(500)) <= {0, 1, 2}
print("empirical frequencies:", freq.round(3))
print("✅ temperature, top-k and top-p work")

## Experiment
1. **Shapes of the distribution.** The plot shows the playground's 8 logits after different temperatures and top-p. Change the settings and re-run.
2. **Repetition.** Generate 40 tokens from the tiny model greedily and with sampling, and count repeated 3-word phrases. The model is untrained, so all of it is nonsense, but greedy decoding already tends to recycle the same phrases; in real models greedy loops can be dramatic. Try other seeds and temperatures.

In [ ]:
settings = [(0.5, None), (1.0, None), (1.5, None), (1.5, 0.9)]
fig, axes = plt.subplots(1, len(settings), figsize=(3.3 * len(settings), 2.8), sharey=True)
for ax, (T, p) in zip(axes, settings):
    probs = apply_temperature(logits, T)
    if p is not None:
        probs = top_p_filter(probs, p)
    ax.bar(range(len(probs)), probs, color="tab:orange")
    ax.set_title(f"T = {T}" + (f", top-p {p}" if p else ""))
    ax.set_xticks(range(len(probs)))
axes[0].set_ylabel("probability")
plt.tight_layout(); plt.show()

def generate_sampled(prompt, n_new, rng, **kw):
    cache = new_cache()
    for i, t in enumerate(prompt):
        logits_ = forward_step(t, i, cache)
    seq = list(prompt)
    for _ in range(n_new):
        seq.append(sample(logits_, rng, **kw))
        logits_ = forward_step(seq[-1], len(seq) - 1, cache)
    return seq

def repeated_trigrams(seq):
    grams = [tuple(seq[i:i + 3]) for i in range(len(seq) - 2)]
    return len(grams) - len(set(grams))      # how many 3-grams are repeats of an earlier one

rng = np.random.default_rng(1)
for label, kw in [("greedy      ", dict(T=0)), ("T=1.0       ", dict(T=1.0)), ("T=0.8, p=0.9", dict(T=0.8, p=0.9))]:
    gen = generate_sampled(prompt, 40, rng, **kw)[len(prompt):]
    print(f"{label}: repeated 3-grams = {repeated_trigrams(gen):2d} | {words(gen)}")

**Think about it** (answers in the Solutions section):
1. Why do we never need to cache the *queries*?
2. Our `forward_step` prefill feeds the prompt one token at a time. Why do real systems process the prompt in one parallel pass instead, and why is that still exact?
3. Why is the cached result *exactly* equal (up to float rounding) rather than an approximation?

---
## Solutions
Only run these if you're stuck. They redefine the functions; re-run the check cells afterwards.

In [ ]:
def generate_naive(prompt, n_new):
    seq = list(prompt)
    for _ in range(n_new):
        logits = forward(seq)[-1]
        seq.append(int(np.argmax(logits)))
    return seq

In [ ]:
def new_cache():
    return [{"k": np.zeros((H, 0, DK)), "v": np.zeros((H, 0, DK))} for _ in range(L)]

def forward_step(token, pos, cache):
    COUNT["token_passes"] += 1
    x = P["tok"][token] + P["pos"][pos]
    for l, lp in enumerate(P["layers"]):
        a = rmsnorm(x, lp["g1"])
        q = (a @ lp["Wq"]).reshape(H, 1, DK)
        k = (a @ lp["Wk"]).reshape(H, 1, DK)
        v = (a @ lp["Wv"]).reshape(H, 1, DK)
        cache[l]["k"] = np.concatenate([cache[l]["k"], k], axis=1)
        cache[l]["v"] = np.concatenate([cache[l]["v"], v], axis=1)
        w = softmax(q @ cache[l]["k"].transpose(0, 2, 1) / np.sqrt(DK))   # (H, 1, t)
        o = (w @ cache[l]["v"]).reshape(D)
        x = x + o @ lp["Wo"]
        x = x + gelu(rmsnorm(x, lp["g2"]) @ lp["W1"]) @ lp["W2"]
    return rmsnorm(x, P["g_f"]) @ P["W_U"]

In [ ]:
def generate_cached(prompt, n_new):
    cache = new_cache()
    for i, t in enumerate(prompt):
        logits = forward_step(t, i, cache)
    seq = list(prompt)
    for _ in range(n_new):
        nxt = int(np.argmax(logits))
        seq.append(nxt)
        logits = forward_step(nxt, len(seq) - 1, cache)
    return seq

In [ ]:
def kv_cache_bytes(layers, kv_heads, d_head, seq_len, bytes_per=2, batch=1):
    return 2 * layers * kv_heads * d_head * seq_len * bytes_per * batch

In [ ]:
def apply_temperature(logits, T):
    return softmax(np.asarray(logits, dtype=float) / T)

def top_k_filter(probs, k):
    keep = np.argsort(probs)[::-1][:k]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()

def top_p_filter(probs, p):
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    n_keep = int(np.argmax(cum >= p - 1e-12)) + 1
    keep = order[:n_keep]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()

def sample(logits, rng, T=1.0, k=None, p=None):
    if T == 0:
        return int(np.argmax(logits))
    probs = apply_temperature(np.asarray(logits, dtype=float), T)
    if k is not None:
        probs = top_k_filter(probs, k)
    if p is not None:
        probs = top_p_filter(probs, p)
    return int(rng.choice(len(probs), p=probs))

**Think-about-it answers**
1. A query is only used by its own token, at the moment that token is processed. Later tokens attend with *their own* queries to *earlier* keys and values, so old queries are never needed again.
2. A GPU is far more efficient doing one big matrix multiply over all prompt tokens than many tiny ones. It's exact because of the causal mask: in the parallel pass, position $i$ only sees positions $\le i$, which is exactly what it would have seen one step at a time.
3. Causality: token $i$'s keys and values in every layer depend only on tokens $1 \dots i$. Appending tokens never changes them, so reusing them computes exactly the same numbers; only the order of floating-point operations differs.

**Next:** M2, pretraining: where the weights that make these predictions come from.